# NB2 · Hip loads and mass properties

How does leverage change with posture? Compare external load alone, gravity of
the modeled bodies and an example speed-dependent actuator envelope. This study
uses repository geometry and explicit synthetic mass and motor inputs.
[DUK v3 chapter 03](https://app.notion.com/p/4131f3cc03328196a898faaeb9af6530).

In [ ]:
import hashlib
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from duk_core.modeling import load_leg_geometry_json
from duk_core.visualization.notebook import configure_notebook_plots

configure_notebook_plots()

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "robot/config/physical/duk_leg_geometry.json").is_file()
)
geometry_path = repo / "robot/config/physical/duk_leg_geometry.json"
fixture_path = repo / "research/experiments/duk-v3/synthetic.json"
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture["schema_version"] == 1 and fixture["status"] == "synthetic_research_only"
print(
    "Working tree dirty:",
    bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=repo)),
)
print(
    "Commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture["source"])


configure_notebook_plots()

In [ ]:
from duk_core.modeling import build_planar_biped_model, leg_jacobian, solve_leg_pose
from duk_core.modeling.hip_load import LinearTorqueSpeed, ideal_symmetric_load_kg
from duk_core.modeling.planar_numeric import NumericPlanarBiped

model = NumericPlanarBiped(build_planar_biped_model(geometry), fixture["planar"])
settings = fixture["hip_envelope"]
motor = LinearTorqueSpeed(
    settings["stall_torque_N_m"],
    settings["no_load_speed_rad_s"],
    settings["efficiency"],
)
angles = np.linspace(75, 130, 56)
q_samples = np.deg2rad(angles)
gravity = model.parameters["g_m_s2"]
force = model.masses.sum() * gravity / 2
axle_load = [[0.0, force], [0.0, force]]
lever = np.array(
    [
        leg_jacobian(geometry, solve_leg_pose(geometry, q)).d_o[1] / 1000
        for q in q_samples
    ]
)
effort = np.array(
    [model.static_hip_effort([0, 0, 0, q, q, 0, 0], axle_load) for q in q_samples]
)
equivalent = [model.equivalent_upper_body(q) for q in q_samples]
payload = [ideal_symmetric_load_kg(j, motor.joint_limit(0), gravity) for j in lever]
test_speed = 2.0
at_speed = motor.joint_limit(test_speed)
saturated = angles[np.max(abs(effort), axis=1) > at_speed]

np.testing.assert_allclose(10 * lever[[15, 45]], [1.333, 2.769], atol=0.0005, rtol=0)
np.testing.assert_allclose(effort[:, 0], effort[:, 1], atol=1e-10, rtol=0)
assert all(e.length_m > 0 and e.inertia_about_center_kg_m2 > 0 for e in equivalent)
print(
    "Synthetic near-static poses exceeding the illustrative envelope at 2 rad/s:",
    saturated,
)
print("Upper mass (kg):", equivalent[0].mass_kg)

In [ ]:
from IPython.display import HTML, display

from duk_core.visualization.notebook import result_table

fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
demand = np.max(abs(effort), axis=1)
axes[0, 0].plot(angles, abs(force * lever), label="Axle load only")
axes[0, 0].plot(angles, demand, label="With modeled gravity")
axes[0, 0].axhline(at_speed, color="0.3", ls="--", label="Bound at 2 rad/s")
axes[0, 0].fill_between(
    angles, at_speed, demand, where=demand > at_speed, color="#D55E00", alpha=0.18
)
axes[0, 0].set(ylabel="Joint moment (N·m)", title="Static demand vs illustrative bound")
axes[0, 0].legend(fontsize=8)
axes[0, 1].plot(angles, payload)
axes[0, 1].set(ylabel="Ideal centered load (kg)", title="Massless-leg load estimate")
axes[0, 2].plot(angles, abs(lever) * motor.no_load_speed)
axes[0, 2].set(ylabel="Height speed (m/s)", title="No-load kinematic speed")
axes[1, 0].plot(angles, [e.center_mm[0] for e in equivalent], label="x")
axes[1, 0].plot(angles, [e.center_mm[1] for e in equivalent], label="z")
axes[1, 0].set(ylabel="Upper CoM (mm)", title="Center of mass")
axes[1, 0].legend()
axes[1, 1].plot(angles, [e.length_m for e in equivalent])
axes[1, 1].set(ylabel="Axle–CoM distance (m)", title="Equivalent pendulum length")
axes[1, 2].plot(angles, [e.inertia_about_center_kg_m2 for e in equivalent])
axes[1, 2].set(ylabel="CoM inertia (kg·m²)", title="Upper-body inertia")
for ax in axes.flat:
    ax.set_xlabel("Linkage angle (deg)")
plt.show()


display(
    HTML(
        result_table(
            [
                {
                    "q (deg)": angles[i],
                    "Required (N·m)": demand[i],
                    "Bound (N·m)": at_speed,
                    "Ideal load (kg)": payload[i],
                    "Static screen": "PASS" if demand[i] <= at_speed else "FAIL",
                }
                for i in (0, 15, 35, 45, 55)
            ],
            caption="Synthetic inputs · static demand screened at 2 rad/s (not a dynamic load test)",
        )
    )
)

**Gate.** The two 10 N leverage examples match the v3 reference, the symmetric
effort check passes and the upper-body reduction produces positive mass
properties. The speed-envelope overlay uses the static torque demand to screen
postures. NB5 computes the dynamic joint effort along a moving trajectory.

## Corrected handbook examples: 2 N·m versus 3.5 kg

[Chapter 03, corrected 2026-11-11](https://app.notion.com/p/4131f3cc03328196a898faaeb9af6530)
compares two separate illustrative assumptions: 2 N·m per hip, and the
3.5 kg upper mass of chapter 06. This is a massless-leg, quasistatic screen
using |J_Oy| and equal load sharing. It excludes link gravity, horizontal
force, acceleration and wheel mass. The motor fixture above is a different
synthetic example; neither example identifies a real DUK motor limit.

JᵀF is the external generalized load. Static motor equilibrium is
Q_motor = dV/dq − JᵀF, as implemented by `static_hip_effort`.

In [ ]:
handbook = fixture["handbook_linear_reference"]
illustrative_bound = 2.0  # chapter 03 example, N·m per hip
illustrative_mass = handbook["upper_mass_kg"]
illustrative_gravity = handbook["gravity_m_s2"]
static_rows = []
for angle in (75, 90, 120, 130):
    local = solve_leg_pose(geometry, np.deg2rad(angle))
    jy = leg_jacobian(geometry, local).d_o[1] / 1000
    ideal_mass = ideal_symmetric_load_kg(jy, illustrative_bound, illustrative_gravity)
    required = illustrative_mass * illustrative_gravity * abs(jy) / 2
    static_rows.append(
        {
            "q (deg)": angle,
            "Ideal mass at 2 N·m (kg)": ideal_mass,
            "Per-hip effort for 3.5 kg (N·m)": required,
            "Within illustrative 2 N·m": required <= illustrative_bound,
        }
    )
np.testing.assert_allclose(
    [row["Per-hip effort for 3.5 kg (N·m)"] for row in static_rows],
    [1.6, 2.3, 4.8, 8.5],
    atol=0.05,
    rtol=0,
)
np.testing.assert_allclose(
    [static_rows[i]["Ideal mass at 2 N·m (kg)"] for i in (1, 2)],
    [3.06, 1.47],
    atol=0.005,
    rtol=0,
)
display(
    HTML(
        result_table(
            static_rows,
            caption="Separate handbook assumptions · ideal static screen only",
        )
    )
)
fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
ax.plot(
    [row["q (deg)"] for row in static_rows],
    [row["Per-hip effort for 3.5 kg (N·m)"] for row in static_rows],
    "o-",
    label="3.5 kg upper mass",
)
ax.axhline(
    illustrative_bound, ls="--", color="#D55E00", label="Illustrative 2 N·m per hip"
)
ax.set(
    xlabel="Linkage angle (deg)",
    ylabel="Per-hip moment magnitude (N·m)",
    title="3.5 kg example versus 2 N·m per hip",
)
ax.legend()
plt.show()